In [2]:
import os
import sys
import time

import json
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from glob import glob
os.environ.pop('WAYLAND_DISPLAY', None)  # Open3D's GLFW window fails to open under native Wayland on this machine (GLEW init error) -- force XWayland instead
os.environ['XDG_SESSION_TYPE'] = 'x11'
import open3d as o3d
import matplotlib.pyplot as plt

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# View segmentation results

In [3]:
import numpy as np
import open3d as o3d
from sklearn.cluster import DBSCAN
import matplotlib.pyplot as plt

# Load data
point_file = 'outputs/predicted_3.txt'  # was 'C:/Users/mmdl/Desktop/test_data/predicted_3.txt' -- output of PointCNN_Inference.ipynb
cloud = np.loadtxt(point_file)
point_cloud = np.array(cloud[:, :3])
truth_label = np.array(cloud[:, 3])

point_cloud = point_cloud.reshape(-1, 3)

# Split by label
points_label0 = point_cloud[truth_label == 0]
points_label1 = point_cloud[truth_label == 1]

# Run DBSCAN on the label-0 points
dbscan = DBSCAN(eps=2.0, min_samples=1000)  # tune as needed
labels_dbscan = dbscan.fit_predict(points_label0)

# Remove outliers (label = -1)
inlier_mask = labels_dbscan != -1
points_label0 = points_label0[inlier_mask]

pcd_label0 = o3d.geometry.PointCloud()
pcd_label0.points = o3d.utility.Vector3dVector(points_label0[:]) 
pcd_label0.paint_uniform_color([0, 0, 1])  # blue (RGB)

pcd_label1 = o3d.geometry.PointCloud()
pcd_label1.points = o3d.utility.Vector3dVector(points_label1[:])  
pcd_label1.paint_uniform_color([1, 0, 0])  # red (RGB)

# Merge point clouds
combined_pcd = pcd_label0 + pcd_label1

# Visualize
vis = o3d.visualization.Visualizer()
vis.create_window()
vis.add_geometry(combined_pcd)

# Set up the coordinate axes
axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=10)
#vis.add_geometry(axis)

# Set the viewpoint and parameters
view_ctl = vis.get_view_control()
view_ctl.set_up([0, 0, 1])  # Z axis points up
view_ctl.set_lookat([0, 0, 0])
view_ctl.set_front([1, 1, 1])
# 
vis.run()
vis.destroy_window()

# Voxel filtering of the workpiece surface point cloud

In [1]:
from mpl_toolkits.mplot3d import Axes3D
import scipy.ndimage as ndimage

time3 = time.time()

#points_label1 = largest_cluster_points
weld_bead = points_label1
work_piece = points_label0

fig = plt.figure()  
ax = Axes3D(fig)   
fig.add_axes(ax)
# ax.set_zlim([-20, 20])
ax.set_xlabel("X(mm)")
ax.set_ylabel("Y(mm)")
ax.set_zlabel("Z(mm)")
ax.scatter(work_piece[:, 0], work_piece[:, 1], work_piece[:, 2], c ='blue', s=10)
plt.show()

# Get the min/max of x, y, z
x_min, x_max = np.min(work_piece[:, 0]), np.max(work_piece[:, 0])
y_min, y_max = np.min(work_piece[:, 1]), np.max(work_piece[:, 1])
z_min, z_max = np.min(work_piece[:, 2]), np.max(work_piece[:, 2])

size_v = 1.5  # voxel size
print("Voxel size =", size_v)

# # Compute the voxel grid size
# d_x = (x_max - x_min) / size_v
# d_y = (y_max - y_min) / size_v
# d_z = (z_max - z_min) / size_v

# h = {}  # dict mapping voxel index -> point indices
# for i in range(len(work_piece)):
#     hx = np.floor((work_piece[i, 0] - x_min) / size_v)
#     hy = np.floor((work_piece[i, 1] - y_min) / size_v)
#     hz = np.floor((work_piece[i, 2] - z_min) / size_v)
#     voxel_index = (hx, hy, hz)  # voxel index
    
#     if voxel_index not in h:
#         h[voxel_index] = []
#     h[voxel_index].append(i)

# # Randomly pick one point from each voxel
# filtered_points = [work_piece[random.choice(indices)] for indices in h.values()]

# filtered_points = np.array(filtered_points, dtype=np.float64)
# print(f"{len(filtered_points)} points in total.")

# work_piece = filtered_points  # update work_piece

filtered_points = []
tempx = []
tempy = []
tempz = []
for i in range(0, len(work_piece)):
    tempx.append(work_piece[i][0])       # collect the x values of each point
    tempy.append(work_piece[i][1])       # collect the y values of each point
    tempz.append(work_piece[i][2])       # collect the z values of each point
    
x_max = np.max(tempx)               # min/max of x, y, z
y_max = np.max(tempy)
z_max = np.max(tempz)
x_min = np.min(tempx)
y_min = np.min(tempy)
z_min = np.min(tempz)

size_v =2 #5       
print("Voxel size =", str(size_v))

d_x = (x_max-x_min)/size_v    # number of voxels needed to cover the whole range, e.g. max=10 min=0 voxel size=1 -> (10-0)/1=10 voxels
d_y = (y_max-y_min)/size_v
d_z = (z_max-z_min)/size_v

h = []  # stores the voxel index (linear) for each point
for i in range(0, len(work_piece)):
    hx = np.floor((work_piece[i][0] - x_min) / size_v)   # floor -> project onto the X grid; subtract min to get the voxel index
    hy = np.floor((work_piece[i][1] - y_min) / size_v)   # project onto the Y grid
    hz = np.floor((work_piece[i][2] - z_min) / size_v)   # project onto the Z grid
    h.append(hx + hy*d_x + hz*d_x*d_y)              # ****  linear index

h = np.array(h)
index_h = np.argsort(h)   # sort the h array
h_sorted = h[index_h]     # store the sorted elements in h_sorted
count = 0
np.seterr(divide='ignore', invalid='ignore')  # skip warnings, e.g. division by zero

for i in range(0, len(h_sorted)-1):
    if h_sorted[i] == h_sorted[i+1]:       # same voxel -> skip, move to the next iteration
        continue
    index_point = index_h[count:i+1]
    x_temp = []
    y_temp = []
    z_temp = []
    for p in index_point:
        x_temp.append(work_piece[p][0])         
        y_temp.append(work_piece[p][1])
        z_temp.append(work_piece[p][2])
    filtered_points.append([np.mean(x_temp), np.mean(y_temp), np.mean(z_temp)])   # average x, y, z to get the centroid
    count = i
    
filtered_points = np.array(filtered_points, dtype=np.float64)

x_f = []
y_f = []
z_f = []

for i in range(0, len(filtered_points)):
    x_f.append(filtered_points[i][0])
    y_f.append(filtered_points[i][1])
    z_f.append(filtered_points[i][2])

while (0 in x_f):                  # remove the values that just became 0
    x_f.remove(0)
    y_f.remove(0)
    z_f.remove(0)

print(str(len(x_f)), "points in total.")

# fig = plt.figure()    # create a figure object
# ax = Axes3D(fig)   # create a 3D axes and attach it to fig
# fig.add_axes(ax)

# ax.set_zlim([-20, 20])
# ax.set_xlabel("X(mm)")
# ax.set_ylabel("Y(mm)")
# ax.set_zlabel("Z(mm)")
# ax.scatter(x_f, y_f, z_f, c ='black')
# plt.show()

work_piece = np.array(list(zip(x_f, y_f, z_f)))

point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(work_piece)

# Statistical outlier removal
cl, ind = point_cloud.remove_statistical_outlier(nb_neighbors=5, std_ratio=1.5)
work_piece = np.asarray(cl.points)

fig = plt.figure()  
ax = Axes3D(fig)   
fig.add_axes(ax)
# ax.set_zlim([-20, 20])
ax.set_xlabel("X(mm)")
ax.set_ylabel("Y(mm)")
ax.set_zlabel("Z(mm)")
ax.scatter(work_piece[:, 0], work_piece[:, 1], work_piece[:, 2], c ='blue', s=10)
plt.show()

time4 = time.time()
t2 = np.round((time4 - time3), 4)

print("Voxel filtering time: ", t2, "s")

NameError: name 'time' is not defined

In [ ]:
from scipy.interpolate import SmoothBivariateSpline
from scipy.spatial import KDTree

time5 = time.time()

x = work_piece[:, 0]
y = work_piece[:, 1]
z = work_piece[:, 2]

#-------------------------------------------------------------------------------------------------------------------

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

# Fit the surface with a smoothing bivariate spline
spline = SmoothBivariateSpline(x, y, z, s = 50)

# Build a grid to evaluate the surface on
X, Y = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = spline.ev(X.ravel(), Y.ravel()).reshape(300, 300)

# Compute the normal vectors (via the partial derivatives in X and Y)
dZ_dx = spline.ev(X.ravel(), Y.ravel(), dx=1, dy=0).reshape(300, 300)
dZ_dy = spline.ev(X.ravel(), Y.ravel(), dx=0, dy=1).reshape(300, 300)
normals = np.dstack((-dZ_dx, -dZ_dy, np.ones_like(Z)))
normals /= np.linalg.norm(normals, axis=2, keepdims=True)  # normalize the normal vectors

weld_bead_tree = KDTree(weld_bead)

# Set the offset step size and stopping condition
offset_distance = 0.1  # distance per offset step
max_offset = 100       # maximum offset distance (avoids an infinite loop)

# Offset the surface and check for collisions
current_offset = 1
last_offset_point = []  # stores the last offset points

while current_offset < max_offset:
    # Compute the offset Z values from the normal vectors
    Z_offset = Z + current_offset * normals[..., 2]
    X_offset = X + current_offset * normals[..., 0]
    Y_offset = Y + current_offset * normals[..., 1]
    
    # Compute each offset point's minimum distance to the weld_bead point cloud
    surface_points = np.column_stack((X_offset.ravel(), Y_offset.ravel(), Z_offset.ravel()))
    dist, _ = weld_bead_tree.query(surface_points)
    
    # Check whether every point has cleared the weld_bead point cloud
    if np.all(dist > offset_distance):
        break  # no collision, stop offsetting
    
    # Record the last offset points
    last_offset_point = surface_points[dist <= offset_distance]
    current_offset += offset_distance

# Compute the shortest distance from the original surface to the offset surface
original_points = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
offset_points = np.column_stack((X_offset.ravel(), Y_offset.ravel(), Z_offset.ravel()))

original_tree = KDTree(original_points)

# Find the closest point pair
if len(last_offset_point) > 0:
    # Check the last offset points
    last_offset_point = np.array(last_offset_point)
    
    # Find the nearest point in weld_bead
    weld_bead_distances, nearest_indices = weld_bead_tree.query(last_offset_point)
    closest_weld_bead_points = weld_bead[nearest_indices]

    # Compute that point's shortest distance to the original surface
    closest_original_distances, closest_original_indices = original_tree.query(closest_weld_bead_points)

    # Get the shortest distance
    min_distance = np.round(closest_original_distances.min(), 3)
    
    #print("Last weld_bead point the offset surface touched:", closest_weld_bead_points)
    print("Weld bead height:", min_distance, "mm")

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x, y, z, c='blue', s=5)  # original surface points
# ax.scatter(weld_bead[:, 0], weld_bead[:, 1], weld_bead[:, 2], c='red', s=5)  # weld seam points
ax.plot_surface(X, Y, Z, color='r', alpha=0.5)  # original surface
# ax.plot_surface(X_offset, Y_offset, Z_offset, color='g', alpha=0.5)  # offset surface
ax.set_xlabel('X(mm)')
ax.set_ylabel('Y(mm)')
ax.set_zlabel('Z(mm)')
ax.set_zlim([-20, 40])
plt.title('B-Spline Surface Fitting with Minimum Distance to Weld Bead')
plt.show()

time6 = time.time()
t3 = np.round((time6 - time5), 4)

print("Weld bead height computation time: ", t3, "s")

In [ ]:
from scipy.spatial import Delaunay
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(work_piece)

# 2. Build a triangle mesh over the fitted surface
vertices = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
tri = Delaunay(vertices[:, :2])  # triangulate using only x, y
triangles = tri.simplices  # triangle indices

# mesh = o3d.geometry.TriangleMesh()
# mesh.vertices = o3d.utility.Vector3dVector(vertices)
# mesh.triangles = o3d.utility.Vector3iVector(triangles)
# mesh.compute_vertex_normals()

# # 3. Visualize
# o3d.visualization.draw_geometries([pcd, mesh])

# Point cloud rasterization and skeletonization

In [ ]:
from skimage.morphology import skeletonize, dilation, erosion, disk, opening, convex_hull_image
from skimage.filters import gaussian
import skimage
import cv2

time7 = time.time()

# Create a blank (150, 150) image
image = np.zeros((150, 150), dtype=np.uint8)

# Map the point cloud into the image range
x_mapped = np.clip(((weld_bead[:, 0] - min(weld_bead[:, 0]))).astype(int), 0, image.shape[0] - 1)
y_mapped = np.clip(((weld_bead[:, 1] - min(weld_bead[:, 1]))).astype(int), 0, image.shape[1] - 1)

# Set the mapped points to 255 in the image
image[x_mapped, y_mapped] = 255

selem1 = disk(1)
selem2 = disk(2)
ero_image = erosion(image, selem1)
ero_image = dilation(ero_image, selem1)
ero_image = erosion(ero_image, selem1)
ero_image = dilation(ero_image, selem1)
ero_image = erosion(ero_image, selem2)
ero_image = dilation(ero_image, selem2)
ero_image = erosion(ero_image, selem2)

# Find contours
contours, _ = cv2.findContours(ero_image, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# Create a blank image to hold the result
smoothed_image = np.zeros_like(ero_image)

# epsilon ratio -- tune this to adjust the smoothing strength
epsilon_ratio = 0.003

# Walk all contours and smooth them
for contour in contours:
    epsilon = epsilon_ratio * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    cv2.drawContours(smoothed_image, [approx], -1, (255), thickness=cv2.FILLED)

# Smooth the contour with an opening operation
opened_image = opening(smoothed_image, selem1)

# Skeletonize
# skeleton = skeletonize(opened_image >0.9)
skeleton = skeletonize(opened_image, method='lee')
# skeleton = skeletonize_3d(opened_image)
# skeleton = skimage.morphology.medial_axis(opened_image) 

# Plot the original binary image, dilated image, eroded image, smoothed image, and skeleton
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharex=True)
ax = axes.ravel()

ax[0].imshow(image, cmap=plt.cm.gray)
ax[0].set_title('Original Image')

ax[1].imshow(ero_image, cmap=plt.cm.gray)
ax[1].set_title('Processed Image')

ax[2].imshow(smoothed_image, cmap=plt.cm.gray)
ax[2].set_title('Smoothed Image')

ax[3].imshow(skeleton, cmap=plt.cm.gray)
ax[3].set_title('Skeleton')
plt.show()

time8 = time.time()
t4 = np.round((time8 - time7), 4)

print("Point cloud rasterization time: ", t4, "s")

# Convert skeleton pixels to a point cloud

In [ ]:
from scipy.spatial import distance

time9 = time.time()

skeleton_points = []

for y in range(skeleton.shape[0]):
    for x in range(skeleton.shape[1]):
        # if the pixel is nonzero, add its coordinates to the point data
        if skeleton[y, x]:
            skeleton_points.append([x, y])

skeleton_points = np.array(skeleton_points)
print("Skeleton point cloud shape:", skeleton_points.shape)

x_set = min(weld_bead[:, 0]) - min(image[:, 0]) 
y_set = min(weld_bead[:, 1]) - min(image[:, 1])
x_path = skeleton_points[:, 1] + x_set
y_path = skeleton_points[:, 0] + y_set

path_2D = []
for i in range(len(x_path)):
    path_2D.append([x_path[i], y_path[i]])
    
plt.scatter(x_path, y_path, c = 'blue', label = 'Path points(Skeleton)')
plt.scatter(weld_bead[:, 0], weld_bead[:, 1], c = 'red', alpha = 0.003, label = 'Weld bead')
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.show

# Path point ordering algorithm

In [ ]:
from scipy.spatial import distance_matrix

points = np.array(path_2D)
def calculate_total_distance(points, order):
    distances = distance_matrix(points[order], points[order])
    total_distance = sum(distances[i, i+1] for i in range(len(order) - 1))
    return total_distance

def reorder_weld_bead(points, start_index):
    num_points = len(points)
    distances = distance_matrix(points, points)
    reordered_indices = [start_index]  # Start with the given start point
    remaining_indices = set(range(num_points))  # Indices of points not yet reordered
    remaining_indices.remove(start_index)

    while remaining_indices:
        last_index = reordered_indices[-1]
        nearest_index = min(remaining_indices, key=lambda x: distances[last_index, x])
        reordered_indices.append(nearest_index)
        remaining_indices.remove(nearest_index)

    return reordered_indices

# Initialize the minimum distance
min_total_distance = float('inf')
best_order = None

# Try every point as the start point
for i in range(len(points)):
    current_order = reorder_weld_bead(points, i)
    current_distance = calculate_total_distance(points, current_order)
    
    if current_distance < min_total_distance:
        min_total_distance = current_distance
        best_order = current_order

# Reorder the points using the best ordering found
reordered_points = points[best_order]

plt.figure(figsize=(10, 5))
plt.subplot(121)
plt.scatter(points[0, 0], points[0, 1], c='red', label='First Point')
plt.scatter(points[1:-1, 0], points[1:-1, 1], c='orange', alpha=1, label='Middle Points')
plt.scatter(points[-1, 0], points[-1, 1], c='green', label='Last Point')
plt.title('Original Point Cloud')
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.legend()

plt.subplot(122)
plt.scatter(reordered_points[0, 0], reordered_points[0, 1], c='red', label='First Point')
plt.scatter(reordered_points[1:-1, 0], reordered_points[1:-1, 1], c='orange', alpha=1, label='Middle Points')
plt.scatter(reordered_points[-1, 0], reordered_points[-1, 1], c='green', label='Last Point')
plt.title('Reordered Point Cloud')
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.legend()
plt.show()
path_2D = reordered_points.astype(int)

# path_2D = path_2D[::-1]    # reverse the path


from scipy.ndimage import gaussian_filter1d

# Gaussian-smooth the path
def smooth_path(path, dimension, sigma):
    smoothed_path = np.zeros_like(path)
    for i in range(dimension):  
        smoothed_path[:, i] = gaussian_filter1d(path[:, i], sigma=sigma)
    return smoothed_path

# path_2D = smooth_path(path_2D, 2, 0.5)
# plt.scatter(path_2D[:, 0], path_2D[:, 1], c='orange')
# plt.show()

time10 = time.time()
t5 = np.round((time10 - time9), 4)

print("Path point ordering time: ", t5, "s")

# B-spline curve fitting + Douglas-Peucker algorithm

In [ ]:
import scipy.interpolate as si
import math

time11 = time.time()

tck, u = si.splprep(reordered_points.T, s=2, k=3) #s=5
u = np.linspace(0, 1, 200)
# B-spline curve fitting
t = np.linspace(0, 1, len(reordered_points) - 2, endpoint=True)
t = np.pad(t, (3, 3), mode='constant', constant_values=(0, 1))
# tck = [t, [reordered_points[:, 0], path_2D[:, 1]], 3]
# u = np.linspace(0, 1, 200)
spline_points = np.array(si.splev(u, tck)).T

# Douglas-Peucker algorithm
def douglas_peucker(points, epsilon):
    def perpendicular_distance(point, start, end):
        if np.array_equal(start, end):
            return np.linalg.norm(point - start)
        return np.abs(np.cross(end - start, start - point)) / np.linalg.norm(end - start)

    dmax = 0.0
    index = 0
    end = len(points)
    for i in range(1, end - 1):
        d = perpendicular_distance(points[i], points[0], points[-1])
        if d > dmax:
            index = i
            dmax = d
    if dmax > epsilon:
        rec_results1 = douglas_peucker(points[:index + 1], epsilon)
        rec_results2 = douglas_peucker(points[index:], epsilon)
        return np.vstack((rec_results1[:-1], rec_results2))
    else:
        return np.array([points[0], points[-1]])

epsilon = 1  #1.75
simplified_points = douglas_peucker(spline_points, epsilon)

def uniform_sampling(points, num_samples):
    cumulative_lengths = np.cumsum(np.r_[0, np.sqrt(np.sum(np.diff(points, axis=0) ** 2, axis=1))])
    uniform_samples = np.linspace(0, cumulative_lengths[-1], num_samples)
    return np.vstack([np.interp(uniform_samples, cumulative_lengths, points[:, i]) for i in range(points.shape[1])]).T

# Set the number of uniform samples needed
num_samples = math.ceil(min_total_distance*3)

sampled_points = uniform_sampling(simplified_points, num_samples)
print("num_samples: ", num_samples)

def extrapolate_points_3d(sampled_points, num_extrapolate, num_reference=15):
    num_extrapolate += 1  # extra layer to include the extrapolated point

    # Make sure the reference point count doesn't exceed the point set length
    num_reference = min(num_reference, len(sampled_points) - 1)

    # Compute the trend vector at the start (average of the nearest num_reference points)
    start_directions = np.diff(sampled_points[:num_reference + 1], axis=0)
    start_vector = np.mean(start_directions, axis=0)

    # Compute the trend vector at the end (average of the nearest num_reference points)
    end_directions = np.diff(sampled_points[-(num_reference + 1):], axis=0)
    end_vector = np.mean(end_directions, axis=0)

    # Extrapolate along the trend vector
    start_extrapolated_points = [sampled_points[0] - i * start_vector for i in range(1, num_extrapolate + 1)]
    end_extrapolated_points = [sampled_points[-1] + i * end_vector for i in range(1, num_extrapolate + 1)]

    # Merge the extrapolated points with the original points
    new_points = np.vstack(start_extrapolated_points[::-1] + [sampled_points] + end_extrapolated_points)
    return new_points

sampled_points = np.array(extrapolate_points_3d(sampled_points, 40))  # set how many path points to extrapolate, 55

def calculate_angle_difference(points_2d):
    points_2d = np.array(points_2d, dtype=float)
    
    if points_2d.shape[1] != 2:
        raise ValueError("points_2d must be a 2D numpy array with 2 columns representing 2D coordinates.")
    
    def angle_between(v1, v2):
        """Calculate the angle between two vectors and determine the direction of rotation."""
        # Calculate the dot product and norms
        dot_product = np.dot(v1, v2)
        norm_v1 = np.linalg.norm(v1)
        norm_v2 = np.linalg.norm(v2)
        
        # Calculate the cosine of the angle
        cos_theta = dot_product / (norm_v1 * norm_v2)
        angle = np.arccos(np.clip(cos_theta, -1.0, 1.0))  # Clamp cos_theta to avoid numerical issues
        
        # Determine the direction of rotation using the cross product (in 2D this is just a scalar)
        cross_product = v1[0] * v2[1] - v1[1] * v2[0]
        if cross_product < 0:
            angle = -angle
        
        return np.degrees(angle)
    
    angle_differences = []   
    for i in range(1, len(points_2d)):
        # Compute the direction vectors
        vector_prev = points_2d[i - 1] - points_2d[i - 2] if i > 1 else points_2d[i] - points_2d[i - 1]
        vector_curr = points_2d[i] - points_2d[i - 1]
        
        # Calculate the angle between these vectors
        angle = angle_between(vector_prev, vector_curr)
        angle_differences.append(angle)
    
    return angle_differences

angle_diffs = calculate_angle_difference(sampled_points)
angle_diffs = np.round(angle_diffs, 4)

sampled_points = sampled_points[1:-1, :]
angle_diffs = angle_diffs[:-1]

plt.plot(spline_points[:, 0], spline_points[:, 1], color='black', label='B-spline')
plt.scatter(sampled_points[:, 0], sampled_points[:, 1], color='blue', alpha = 0.05, label='Sampled Points')
# plt.scatter(path_2D[:, 0], path_2D[:, 1], color='blue', label='path_2D', alpha=0.5)
# plt.scatter(reordered_points[:, 0], reordered_points[:, 1], color='orange', label='Path Points', alpha=0.5)
plt.legend()
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.show()

def vector_from_a_to_b(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    vector = b - a
    return vector

def angle_with_reference(vector, reference_vector):
    vector = np.array(vector, dtype=float)
    reference_vector = np.array(reference_vector, dtype=float)
    
    # Compute the angle between the two vectors
    dot_product = np.dot(vector, reference_vector)
    norm_vector = np.linalg.norm(vector)
    norm_reference_vector = np.linalg.norm(reference_vector)
    cos_theta = dot_product / (norm_vector * norm_reference_vector)
    angle = np.arccos(np.clip(cos_theta, -1.0, 1.0))
    cross_product = np.cross(np.append(reference_vector, 0), np.append(vector, 0))
    
    if cross_product[2] < 0:
        angle = -angle
    
    return np.degrees(angle)

a = [sampled_points[0, 0], sampled_points[0, 1]]
b = [sampled_points[1, 0], sampled_points[1, 1]]

# Compute the vector
vector = vector_from_a_to_b(a, b)

reference_vector = [1, 0]

# Compute the angle
angle_init = angle_with_reference(vector, reference_vector)
angle_init = np.round(angle_init, 4)
print("angle_init: ", angle_init)

time12 = time.time()
t6 = np.round((time12 - time11), 4)

print("Path fitting and simplification time: ", t6, "s")

In [ ]:
plt.plot(spline_points[:, 0], spline_points[:, 1], color='black', label='B-spline')
# plt.scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', label='Douglas-Peucker', alpha=0.3)
# plt.scatter(path_2D[:, 0], path_2D[:, 1], color='blue', label='path_2D', alpha=0.5)
plt.scatter(reordered_points[:, 0], reordered_points[:, 1], color='orange', label='Path Points', alpha=1)
plt.legend()
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.show()

In [ ]:
sampled_points = smooth_path(sampled_points, 2, 15)

plt.scatter(sampled_points[:, 0], sampled_points[:, 1], color='blue', alpha=1, s=20)
# plt.scatter(path_2D[:, 0], path_2D[:, 1], color='blue', label='path_2D', alpha=0.5)
# plt.scatter(reordered_points[:, 0], reordered_points[:, 1], color='orange', label='Path Points', alpha=0.5)
# plt.legend()
plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.show()

In [ ]:
plt.scatter(sampled_points[:, 0], sampled_points[:, 1], color='blue')
plt.scatter(weld_bead[:, 0], weld_bead[:, 1], c = 'red', alpha = 0.007, label = 'Weld bead')

plt.axis('equal')
plt.xlabel('X(mm)') 
plt.ylabel('Y(mm)')
plt.show()

# Workpiece surface fitting + projecting the 2D path points into 3D path points

In [ ]:
from scipy.spatial import Delaunay
from scipy.interpolate import griddata

time13 = time.time()

points_3D = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))  # reshape to (N, 3)

# Build the Delaunay triangulation
tri = Delaunay(points_3D[:, :2])

# Compute the projected points' height on the surface
# z_projected = griddata(work_piece[:, :2], work_piece[:, 2], sampled_points, method='linear')   #linear
z_projected  = spline.ev(sampled_points[:, 0], sampled_points[:, 1])

# Plot the surface and the original 3D point cloud
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# simplices = tri.simplices
# ax.plot_trisurf(work_piece[:, 0], work_piece[:, 1], work_piece[:, 2], triangles=simplices, color='b', alpha=0.2)
# # ax.scatter(sampled_points[:, 0], sampled_points[:, 1], z_projected, s = 5, c='green', alpha=1)
# ax.scatter(weld_bead[:, 0], weld_bead[:, 1],weld_bead[:, 2], s = 10, c = 'red', alpha=0.005)
# plt.show()

path_3D = []
for i in range(len(sampled_points)):
    path_3D.append([sampled_points[i, 0], sampled_points[i, 1], z_projected[i]])
    
time14 = time.time()
t7 = np.round((time14 - time13), 4)

print("Path point projection time: ", t7, "s")

In [ ]:
def flip_point_cloud(point_cloud, axis='x'):
    # build the rotation matrix, defaults to a 180-degree rotation about the X axis
    if axis == 'x':
        rotation_matrix = np.array([[1,  0,  0],
                                    [0, -1,  0],
                                    [0,  0, -1]])
    elif axis == 'y':
        rotation_matrix = np.array([[-1,  0,  0],
                                    [0,  1,  0],
                                    [0,  0, -1]])
    elif axis == 'z':
        rotation_matrix = np.array([[-1,  0,  0],
                                    [0, -1,  0],
                                    [0,  0,  1]])
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(point_cloud)
    pcd.rotate(rotation_matrix)
    return np.asarray(pcd.points)

rotation_matrix = np.array([[1,  0,  0],
                            [0, -1,  0],
                            [0,  0, -1]])

# Normal vector computation

In [ ]:
time15 = time.time()

path_3D = np.array(path_3D)

def compute_normal(tri, points):
    normals = []
    for point in points:
        # Find the triangle containing the point
        simplex = tri.find_simplex(point[:2])
        if simplex == -1:
            normals.append([0, 0, 1])  # Default normal if point is outside the convex hull
            continue
        vertices = tri.simplices[simplex]
        p0, p1, p2 = points_3D[vertices]
        # Compute the normal vector of the triangle
        v1 = p1 - p0
        v2 = p2 - p0
        normal = np.cross(v1, v2)
        normal = normal / np.linalg.norm(normal)  # Normalize the vector
        normals.append(normal)
    return np.array(normals)

normals = compute_normal(tri, path_3D)
normals = (rotation_matrix @ normals.T).T

time16 = time.time()
t8 = np.round((time16 - time15), 4)

print("Normal vector computation time: ", t8, "s")

In [ ]:
# transformation_matrix = np.array([
#     [1.0, 0.0, 0.0, 0.0],  # no rotation/translation on the x axis
#     [0.0, 1.0, 0.0, 0.0],  # no rotation/translation on the y axis
#     [0.0, 0.0, 1.0, 0.0],  # no rotation/translation on the z axis
#     [0.0, 0.0, 0.0, 1.0]   # homogeneous coordinate
# ])

transformation_matrix = np.array([
     [ 7.19801821e-02,  7.09806443e-03,  9.97380805e-01, -1.96533126e+02],
     [ 1.33786441e-02,  9.99877844e-01, -8.08136126e-03,  7.65496438e+00],
     [-9.97316332e-01,  1.39253007e-02,  7.18764269e-02,  1.98747624e+02],
     [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

rotation_matrix_z = np.array([
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, 1]
])
rotation_matrix_x = np.array([
    [1, 0, 0],
    [0, -1, 0],
    [0, 0, -1]
])

#----------------------------------------------------------------------------------------------------------

# Stack path_3D and work_piece along the same dimension
combined_point_cloud = np.vstack([path_3D, work_piece])

combined_point_cloud = flip_point_cloud(combined_point_cloud, axis='x') # the reconstruction rotated it 180 deg about X, undo that here
# combined_point_cloud = flip_point_cloud(combined_point_cloud, axis='z')
# Convert the point cloud to homogeneous coordinates
combined_point_cloud_homogeneous = np.hstack([combined_point_cloud, np.ones((len(combined_point_cloud), 1))])

# Transform the combined homogeneous coordinates
transformed_point_cloud = np.dot(combined_point_cloud_homogeneous, transformation_matrix.T)

# Convert back to 3D coordinates
transformed_point_cloud = transformed_point_cloud[:, :3]

# Apply the rotation matrix
transformed_point_cloud = np.dot(transformed_point_cloud, rotation_matrix_z.T) # convert into RobotStudio's TCP coordinate frame

# Split back into the original path_3D and work_piece
num_path_3D_points = len(path_3D)
path_3D = transformed_point_cloud[:num_path_3D_points]
work_piece = transformed_point_cloud[num_path_3D_points:]

normals_rotation = transformation_matrix[:3, :3]

normals = np.array(normals)
normals = np.dot(normals, normals_rotation.T)
# normals = np.dot(normals, rotation_matrix_x.T)
normals = np.dot(normals, rotation_matrix_z.T)
# normals = -1*normals


#----------------------------------------------------------------------------------------------------------

# Convert the point cloud data into an Open3D point cloud object
path_pcd = o3d.geometry.PointCloud()
path_pcd.points = o3d.utility.Vector3dVector(path_3D)

work_piece_pcd = o3d.geometry.PointCloud()
work_piece_pcd.points = o3d.utility.Vector3dVector(work_piece)

# Visualize the normal vectors
normals_pcd = o3d.geometry.PointCloud()
normals_pcd.points = o3d.utility.Vector3dVector(path_3D)
normals_pcd.normals = o3d.utility.Vector3dVector(normals)

# Create a coordinate frame (size 20)
coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=20)

# Visualize
# o3d.visualization.draw_geometries([path_pcd, work_piece_pcd, normals_pcd, coordinate_frame])


# Multi-layer grinding path offset

In [ ]:
time17 = time.time()

# Gaussian path-smoothing function
def smooth_path_gaussian(path_3D, sigma):
    smoothed_path = np.zeros_like(path_3D)    
    for i in range(3):
        smoothed_path[:, i] = gaussian_filter1d(path_3D[:, i], sigma=sigma)
    return smoothed_path

def offset_point_cloud(work_piece, path_3D, num_layers, offset_distance, normals, sigma=1):
    # Gaussian-smooth the path
#     path_3D = smooth_path_gaussian(path_3D, sigma)
    pcd_work_piece = o3d.geometry.PointCloud()
    pcd_work_piece.points = o3d.utility.Vector3dVector(work_piece)
    pcd_work_piece.paint_uniform_color([0, 0, 1])

    pcd_path = o3d.geometry.PointCloud()
    pcd_path.points = o3d.utility.Vector3dVector(path_3D)
    pcd_path.paint_uniform_color([1, 0, 0])

    # Get the normal vectors of the projected points
    normals = np.asarray(normals)

    # Build the list that stores each layer's offset path points
    offset_layers = []
    offset_point_clouds = []
    for i in range(1, num_layers + 1):
        # Offset the path point cloud
        offset = path_3D + normals * (offset_distance * i)
        pcd_offset = o3d.geometry.PointCloud()
        pcd_offset.points = o3d.utility.Vector3dVector(offset)
        pcd_offset.paint_uniform_color([1, 0, 0])
        
        # Save this layer's offset path points to the list
        offset_layers.append(offset)  # this layer's offset path points
        
        # Add this layer's point cloud to the visualization
        offset_point_clouds.append(pcd_offset)  

    # Add the surface point cloud to the visualization list too
    all_point_clouds = [pcd_work_piece, pcd_path]
    all_point_clouds.extend(offset_point_clouds)

    # Draw the normal vectors
    lines = []
    colors = []
    points = []
    for i in range(len(path_3D)):
        start_point = path_3D[i]
        end_point = start_point + normals[i] * min_distance  # scale factor to adjust the normal vector length
        points.append(start_point)
        points.append(end_point)
        lines.append([2 * i, 2 * i + 1])
        colors.append([0, 0, 0])  # black for the normal vectors

    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(points)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    line_set.colors = o3d.utility.Vector3dVector(colors)

    all_point_clouds.append(line_set)

    # Visualize
    axis = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=20.0, origin=[0, 0, 0]  # coordinate frame size and origin
    )
    
    vis = o3d.visualization.Visualizer()
    vis.create_window()
    opt = vis.get_render_option()
    #opt.background_color = np.asarray([0, 0, 0])  # set the background color to black
    vis.add_geometry(axis)
    
    for pcd in all_point_clouds:
        vis.add_geometry(pcd)

    vis.run()
    vis.destroy_window()

    # Return each layer's offset path points (as NumPy arrays) and the normals
    offset_np = [np.asarray(pcd.points) for pcd in offset_point_clouds]

    return offset_np, normals, offset_layers  # also return each layer's offset path points

def smooth_normals(normals, sigma=5):
    smoothed_normals = np.zeros_like(normals)
    
    for i in range(3):  # smooth the x, y, z components separately
        smoothed_normals[:, i] = gaussian_filter1d(normals[:, i], sigma=sigma, mode='reflect')

    # avoid division-by-zero from numerical error
    norm = np.linalg.norm(smoothed_normals, axis=1, keepdims=True)
    norm[norm == 0] = 1  # avoid dividing by zero
    smoothed_normals /= norm  # renormalize the smoothed normals

    return smoothed_normals

# Call the function
normals = smooth_normals(normals, sigma=0.75)
normals = smooth_normals(normals, sigma=0.75)
normals = smooth_normals(normals, sigma=0.75)


# path_3D = smooth_path(path_3D, 3, sigma=10)   #7

num_layers = 0       # number of offset layers
offset_distance = min_distance/num_layers  # offset distance

time4 = time.time()

offset_point_clouds, normals, offset_layers = offset_point_cloud(work_piece, path_3D, num_layers, offset_distance, normals)

for i, layer_points in enumerate(offset_layers):
    df = pd.DataFrame(layer_points, columns=['x', 'y', 'z'])
    df = df.iloc[1:].reset_index(drop=True)
    globals()[f'df_path{i + 1}'] = df
    
time18 = time.time()
t9 = np.round((time18 - time17), 4)

print("Multi-layer grinding path offset time: ", t9, "s")

In [ ]:
# path_3D = np.asarray(path_3D)

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(path_3D[:, 0], path_3D[:, 1], path_3D[:, 2], c='blue', s=5)
# ax.plot_surface(X, Y, Z, color='r', alpha=0.5)  # original surface
# ax.set_xlabel('X(mm)')
# ax.set_ylabel('Y(mm)')
# ax.set_zlabel('Z(mm)')
# ax.set_zlim([-20, 40])
# plt.show()

In [ ]:

# plt.scatter(path_3D[:, 2], path_3D[:, 1], c='blue')
# plt.scatter(weld_bead[:, 0], weld_bead[:, 1], c='red', alpha = 0.005)
# plt.axis('equal')
# plt.xlabel('X(mm)') 
# plt.ylabel('Y(mm)')

# plt.show()

In [ ]:

# plt.scatter(path_3D[14:-15, 0], path_3D[14:-15, 1], c='blue')
# plt.scatter(weld_bead[:, 0], weld_bead[:, 1], c='red', alpha = 0.007)
# plt.axis('equal')
# plt.xlabel('X(mm)') 
# plt.ylabel('Y(mm)')

# plt.show()

# Path movement vectors

In [ ]:
time19 = time.time()

# Compute the connecting line between each pair of adjacent points
lines = [[i, i + 1] for i in range(len(path_3D) - 1)]
colors = [[1, 0, 0] for _ in range(len(lines))]  # red

# Compute the vector for each connecting line
vectors = []
for i in range(len(path_3D) - 1):
    start_point = path_3D[i]
    end_point = path_3D[i + 1]
    vector = end_point - start_point
    vectors.append(vector)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(path_3D)

line_set = o3d.geometry.LineSet(
    points=o3d.utility.Vector3dVector(path_3D),
    lines=o3d.utility.Vector2iVector(lines),
)
line_set.colors = o3d.utility.Vector3dVector(colors)

#o3d.visualization.draw_geometries([pcd, line_set])

# Quaternion computation

In [ ]:
import pandas as pd
from scipy.spatial.transform import Rotation as R

path_3D = path_3D[1:]  # drop the first path point so every tangent vector has a matching point
normals = -1*normals[1:]
vectors = np.array(vectors)

# def normal_to_euler(normal_vector):
#     # normalize the normal vector
#     normal_vector = np.array(normal_vector)
#     normal_vector = normal_vector / np.linalg.norm(normal_vector)

#     reference_vector = np.array([0, 0, 1])
#     axis = np.cross(reference_vector, normal_vector)
#     angle = np.arccos(np.dot(reference_vector, normal_vector))

#     if np.linalg.norm(axis) != 0:
#         axis = axis / np.linalg.norm(axis)  # normalize the rotation axis
#         rotation = R.from_rotvec(axis * angle)
#     else:
#         # when the normal is parallel to the reference vector, the rotation axis is zero, so no rotation is needed
#         rotation = R.from_rotvec(np.array([0, 0, 0]))
    
#     # convert the rotation object to Euler angles (assumes 'xyz' order)
#     euler_angles = rotation.as_euler('xyz', degrees=True)
#     return euler_angles

# def euler_to_quaternion(euler_angles):
#     # create the rotation object with a given Euler angle order (assumes 'xyz')
#     rotation = R.from_euler('xyz', euler_angles, degrees=True)
    
#     # convert the rotation object to a quaternion, format [X, Y, Z, W]
#     quaternion = rotation.as_quat()
    
#     # reorder the quaternion to [W, X, Y, Z]
#     quaternion = [quaternion[3], quaternion[0], quaternion[1], quaternion[2]]   
#     return quaternion

# quaternion = []
# for i in range(len(normals)):
#     euler_angles = normal_to_euler(normals[i, :]*-1)
#     qt = euler_to_quaternion(euler_angles)
#     quaternion.append(qt)

# df1 = pd.DataFrame(path_3D, columns=['x', 'y', 'z'])
# df2 = pd.DataFrame(normals, columns=['i', 'j', 'k'])
# df3 = pd.DataFrame(quaternion, columns=['q1', 'q2', 'q3', 'q4'])

# df = pd.concat([df1, df2, df3], axis=1)
# df.iloc[:, :3] += [450, 0, 250]
# df = df.round(4)
# df  

def calculate_quaternions_with_non_orthogonal_normals(path_3D, normals, t_vectors):
    quaternions = []
    
    for t_vector, normal in zip(t_vectors, normals):
        # Step 1: Normalize t_vector
        t_vector = t_vector / np.linalg.norm(t_vector)
        
        # Step 2: Make normal orthogonal to t_vector using Gram-Schmidt process
        normal = normal - np.dot(normal, t_vector) * t_vector  # Subtract the projection of normal onto t_vector
        normal = normal / np.linalg.norm(normal)  # Normalize the adjusted normal
        
        # Step 3: Compute the y-axis using cross product of z-axis (normal) and x-axis (t_vector)
        y_axis = np.cross(normal, t_vector)
        y_axis = y_axis / np.linalg.norm(y_axis)  # Normalize y_axis
        
        # Step 4: Construct the rotation matrix
        rotation_matrix = np.vstack((t_vector, y_axis, normal)).T  # 3x3 rotation matrix
        
        # Step 5: Convert the rotation matrix to a quaternion
        quat = R.from_matrix(rotation_matrix).as_quat()  # Quaternion in [x, y, z, w] format
        
        # Step 6: Reorder quaternion to [w, x, y, z]
        quat_reordered = [quat[3], quat[0], quat[1], quat[2]]
        
        quaternions.append(quat_reordered)

    return np.array(quaternions)


quaternion = calculate_quaternions_with_non_orthogonal_normals(path_3D, normals, vectors)

time20 = time.time()
t10 = np.round((time20 - time19), 4)

print("Robot arm pose computation time: ", t10, "s")

time21 = time.time()

df_path0 = pd.DataFrame(path_3D, columns=['x', 'y', 'z'])
df_normals = pd.DataFrame(normals, columns=['i', 'j', 'k'])
df_quaternion = pd.DataFrame(quaternion, columns=['q1', 'q2', 'q3', 'q4'])
df_0 = pd.concat([df_path0, df_normals, df_quaternion], axis=1)
df_0.iloc[:, :3] += [0, 0, 0]         #[450, 0, 450]
df_0 = df_0.round(4) 

for n, offset_layer in enumerate(offset_layers, start=1):
    df_path_n = pd.DataFrame(offset_layer, columns=['x', 'y', 'z'])
    df_n = pd.concat([df_path_n, df_normals, df_quaternion], axis=1)
    df_n = df_n.iloc[:-1]
    df_n = df_n.round(4)
    df_n.iloc[:, :3] += [0, 0, 0]     #[450, 0, 450]
    globals()[f'df_{n}'] = df_n
    
time22 = time.time()

# Path file generation

In [ ]:
#TCP

movement_speed = 300
grinding_speed = 3
tool_id = 1

#[0.0,0.7071,0.0,0.7071]

robtarget_template = "CONST robtarget p{layer}_{index}:=[[{x},{y},{z}], [{q1},{q2},{q3},{q4}], [0,0,1,0], [9E9,9E9,9E9,9E9,9E9,9E9]];"

offset_layers = []
for i in range(num_layers + 1):
    offset_layers.append(globals()[f'df_{i}'])

with open('D:/robotic_arm/path_points/TCP_path_test.mod', 'w') as f:
    f.write("MODULE Module1\n")
    f.write(f"    PERS tooldata tool{tool_id}:=[TRUE,[[100,0,100],[0.5,0.5,0.5,0.5]],[3,[0,0,3],[1,0,0,0],0,0,0]];\n")
    for layer_idx, df in enumerate(reversed(offset_layers)):
        real_layer_idx = num_layers - layer_idx  
        for i, row in df.iterrows():
            robtarget_str = robtarget_template.format(
                layer=real_layer_idx,  
                index=i,              
                x=row['x'],
                y=row['y'],
                z=row['z'],
                q1=row['q1'],
                q2=row['q2'],
                q3=row['q3'],
                q4=row['q4']
            )
            f.write("    " + robtarget_str + '\n')
    f.write("    PROC main()\n")
    f.write("        ConfJ\\On;\n")
    f.write("        ConfL\\On;\n")
    for layer_idx, df in enumerate(reversed(offset_layers)):
        real_layer_idx = num_layers - layer_idx 
        f.write(f"        MoveAbsJ [[0.00,5.00,5.00,0.00,5.00,0.00],[9E9,9E9,9E9,9E9,9E9,9E9]],v{movement_speed},fine,tool{tool_id};\n")
        for i in range(len(df)):
            movej_str = f"        MoveJ p{real_layer_idx}_{i}, v{grinding_speed}, fine, tool{tool_id};"
            f.write(movej_str + '\n')
        f.write(f"        MoveAbsJ [[0.00,5.00,5.00,0.00,5.00,0.00],[9E9,9E9,9E9,9E9,9E9,9E9]],v{movement_speed},fine,tool{tool_id};\n")
    f.write("    ENDPROC\n")
    f.write("ENDMODULE")
    
time23 = time.time()
t11 = np.round((time23 - time21), 4)

print("TCP path export time: ", t11, "s")
print("Path .mod file generated, total system computation time approx.:", np.round((t1+t2+t3+t4+t5+t6+t7+t8+t9+t10+t11), 4), "s")

In [ ]:
#RTCP

#tooldata orientation 0.5,-0.5,-0.5,-0.5 is RobotMaster's
#tooldata orientation 0.5,-0.5,0.5,0.5 is the orientation the planned path can actually execute

movement_speed = 300
grinding_speed = 3
tool_id = 1
wobj_id = 1

robtarget_template = "CONST robtarget p{layer}_{index}:=[[{x},{y},{z}], [{q1},{q2},{q3},{q4}], [0,0,0,0], [9E9,9E9,9E9,9E9,9E9,9E9]];"

offset_layers = []
for i in range(num_layers + 1):
    offset_layers.append(globals()[f'df_{i}'])

with open('D:/robotic_arm/path_points/RTCP_path_test.mod', 'w') as f:
    f.write("MODULE Module1\n")
    f.write(f"    VAR speeddata v{grinding_speed} := [{grinding_speed}, 10, 5000, 1000];\n")
    f.write(f"    PERS wobjdata wobj{wobj_id}:=[TRUE,TRUE,\"\",[[0.00,0.00,0.00],[0,0,0,1]],[[0,0,0],[1,0,0,0]]];\n")
    f.write(f"    PERS tooldata tool{tool_id}:=[FALSE,[[433,-367.7,137.26],[0.5,-0.5,-0.5,-0.5]],[3,[0,0,3],[1,0,0,0],0,0,0]];\n")
    f.write(f"    PERS tooldata tool2:=[FALSE,[[433,-442.7,215.89],[0.7071,0,0,-0.7071]],[3,[0,0,3],[1,0,0,0],0,0,0]];\n")
    for layer_idx, df in enumerate(reversed(offset_layers)):
        real_layer_idx = num_layers - layer_idx  
        for i, row in df.iterrows():
            robtarget_str = robtarget_template.format(
                layer=real_layer_idx,  
                index=i,              
                x=row['x'],
                y=row['y'],
                z=row['z'],
                q1=row['q1'],
                q2=row['q2'],
                q3=row['q3'],
                q4=row['q4']
            )
            f.write("    " + robtarget_str + '\n')
    f.write("    PROC main()\n")
    f.write("        ConfJ\\On;\n")
    f.write("        ConfL\\On;\n")
    for layer_idx, df in enumerate(reversed(offset_layers)):
        real_layer_idx = num_layers - layer_idx 
        f.write(f"        MoveAbsJ [[0.00,5.00,5.00,0.00,5.00,0.00],[9E9,9E9,9E9,9E9,9E9,9E9]],v{movement_speed},fine,tool{tool_id}\Wobj:=wobj{wobj_id};\n")
        for i in range(len(df)):
            RelTool1_str = f"        MoveJ RelTool(p{real_layer_idx}_{i}, 0, 0, -10), v{movement_speed}, fine, tool1\Wobj:=wobj1;"
            RelTool2_str = f"        MoveJ RelTool(p{real_layer_idx}_{i}, 0, 0, -10), v{grinding_speed}, fine, tool1\Wobj:=wobj1;"
            movej_str = f"        MoveJ p{real_layer_idx}_{i}, v{grinding_speed}, fine, tool{tool_id}\Wobj:=wobj{wobj_id};"
            if i == 0:
                f.write(RelTool1_str + '\n')
            f.write(movej_str + '\n')
            if i == len(df)-1:
                f.write(RelTool2_str + '\n')
        f.write(f"        MoveAbsJ [[0.00,5.00,5.00,0.00,5.00,0.00],[9E9,9E9,9E9,9E9,9E9,9E9]],v{movement_speed},fine,tool{tool_id}\Wobj:=wobj{wobj_id};\n")
    f.write("    ENDPROC\n")
    f.write("ENDMODULE")
    
time24 = time.time()
t12 = np.round((time24 - time23)+(time22 - time21), 4)

print("RTCP path export time: ", t11, "s")
print("Path .mod file generated, total system computation time approx.:", np.round((t1+t2+t3+t4+t5+t6+t7+t8+t9+t10+t12), 4), "s")

In [ ]:
print("Weld bead recognition: ", t1, "s")
print("Voxel filtering: ", t2, "s")
print("Weld bead height computation: ", t3, "s")
print("Point cloud rasterization: ", t4, "s")
print("Path point ordering: ", t5, "s")
print("Path fitting and simplification: ", t6, "s")
print("Path point projection: ", t7, "s")
print("Normal vector computation: ", t8, "s")
print("Multi-layer grinding path offset: ", t9, "s")
print("Robot arm pose computation: ", t10, "s")